<a href="https://colab.research.google.com/github/love-adu/CheckPoint/blob/main/antivaccorgscraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anti-Vaxx Organization List Scraper Type Shiii
By Love da GOAT

What it does:
- Searches DuckDuckGo, Bing, and Wikipedia using a list of queries defined below
- Finds the full text of each article it finds
- Runs spaCy named entity recognition (NER) to extract organization names from article text
- Uses TextBlob sentiment analysis to check whether the context around each org name is negative toward vaccines
- Scores and ranks orgs by how often they appeared and how strong the anti-vax signal was
- Exports everything to CSV and JSON

Also Google isn't used because it blocks bots with CAPTCHAs after a few requests. DuckDuckGo and Bing work without any API keys and give equivalent coverage impo


## install

In [ ]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 lxml pandas spacy textblob duckduckgo-search newspaper3k tqdm --quiet
!{sys.executable} -m spacy download en_core_web_sm --quiet
print('✓ All dependencies installed')

## imports

In [ ]:
import requests, time, re, json
from datetime import datetime
from collections import defaultdict
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from IPython.display import display, HTML, FileLink, clear_output
import spacy
from textblob import TextBlob
from duckduckgo_search import DDGS

# Load spaCy NER model
nlp = spacy.load('en_core_web_sm')

HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; ResearchBot/1.0; Public Health Research)'}

print('✓ NLP pipeline loaded (spaCy NER + TextBlob sentiment)')
print(f'  spaCy model: en_core_web_sm')

## settings

 `MAX_PAGES` and `RESULTS_PER_QUERY` control scrape depth and higher is slower but finds more. `MIN_CONFIDENCE` filters out orgs that only appeared once (do we want this?).

In [ ]:
RESULTS_PER_QUERY  = 10     # results per search query (10 = ~1 page)
MAX_PAGES          = 10     # number of search pages to simulate per query
SCRAPE_ARTICLES    = True   # fetch & NLP-parse full article text
SENTIMENT_THRESHOLD = -0.05 # TextBlob polarity threshold (more negative = stricter)
MIN_CONFIDENCE     = 2      # min times an org must appear to be included
DELAY_SECONDS      = 1.5    # polite delay between requests

# Search queries that are sent to DuckDuckGo + Bing
SEARCH_QUERIES = [
    'anti-vaccine organizations list',
    'vaccine misinformation groups',
    'anti-vaccination movement organizations',
    'organizations opposing mandatory vaccination',
    'vaccine hesitancy advocacy groups',
    'anti-vax nonprofit organizations',
    'vaccine safety skeptics organizations',
    'anti-immunization groups worldwide',
    'groups spreading vaccine misinformation',
    'vaccine freedom organizations',
]

print(f'✓ Config set: {len(SEARCH_QUERIES)} queries × {MAX_PAGES} pages = up to {len(SEARCH_QUERIES)*MAX_PAGES*RESULTS_PER_QUERY} search results')

## search functions

Three sources: DuckDuckGo, Bing, Wikipedia API

In [ ]:
def search_duckduckgo(query, max_results=10):
    """Search DuckDuckGo"""
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append({'title': r.get('title',''), 'url': r.get('href',''),
                                 'snippet': r.get('body',''), 'source': 'duckduckgo'})
    except Exception as e:
        print(f'  [DDG] {e}')
    return results


def search_bing(query, max_results=10):
    """Scrape Bing"""
    results = []
    try:
        for page in range(0, max_results, 10):
            url = f'https://www.bing.com/search?q={requests.utils.quote(query)}&first={page+1}'
            r = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(r.text, 'html.parser')
            for li in soup.select('li.b_algo'):
                title_el = li.select_one('h2 a')
                snip_el  = li.select_one('.b_caption p, .b_algoSlug')
                if title_el:
                    results.append({
                        'title': title_el.get_text(strip=True),
                        'url': title_el.get('href',''),
                        'snippet': snip_el.get_text(strip=True) if snip_el else '',
                        'source': 'bing'
                    })
    except Exception as e:
        print(f'  [Bing] {e}')
    return results


def search_wikipedia(query):
    """Search Wikipedia"""
    results = []
    try:
        url = 'https://en.wikipedia.org/w/api.php'
        params = {'action':'query','list':'search','srsearch':query,'format':'json','srlimit':10}
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        for item in r.json().get('query',{}).get('search',[]):
            results.append({
                'title': item['title'],
                'url': f"https://en.wikipedia.org/wiki/{item['title'].replace(' ','_')}",
                'snippet': BeautifulSoup(item.get('snippet',''), 'html.parser').get_text(),
                'source': 'wikipedia'
            })
    except Exception as e:
        print(f'  [Wikipedia] {e}')
    return results


print('✓ Search functions done (DuckDuckGo + Bing + Wikipedia)')

## NLP for entity extraction + sentiment

spaCy pulls organizations entities out of article text and then each one gets scored based on whether vaccine-related keywords appear in the name or surrounding text, and whether TextBlob reads the context as negative.

In [ ]:
# Keywords that suggest anti-vaccine stance
ANTIVAX_KEYWORDS = [
    'anti-vacc','antivax','anti-vax','vaccine choice','vaccine freedom',
    'vaccine hesitancy','vaccine injury','no mandatory vax','vaccine risk',
    'vaccine danger','stop vaccination','against vaccine','oppose vaccine',
    'vaccine mandate','refuse vaccine','natural immunity','informed consent',
    'vaccine harm','vaccine skeptic','vaccine truth','children health defense',
    'health freedom','medical freedom'
]

ORG_SUFFIXES = [
    'network','coalition','alliance','foundation','institute','center','centre',
    'association','organization','society','group','movement','action','council',
    'committee','union','league','forum','trust','fund','project'
]

def extract_orgs_ner(text):
    """Extract ORG entities using spaCy NER."""
    doc = nlp(text[:100000])  # spaCy limit
    orgs = set()
    for ent in doc.ents:
        if ent.label_ == 'ORG' and len(ent.text) > 5:
            name = ent.text.strip()
            # Filter noise
            if not any(c.isdigit() for c in name) and len(name.split()) >= 2:
                orgs.add(name)
    return orgs


def has_antivax_signal(text):
    """Check if text contains anti-vaccine keywords."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in ANTIVAX_KEYWORDS)


def sentiment_score(text):
    """Return TextBlob polarity score (-1 negative → +1 positive)."""
    try:
        return TextBlob(text[:5000]).sentiment.polarity
    except:
        return 0.0


def score_org_candidate(name, context_text):
    """how likely a named entity is an anti-vaccine org"""
    score = 0
    name_lower = name.lower()
    ctx_lower = context_text.lower()

    # Org-like suffix in name
    if any(s in name_lower for s in ORG_SUFFIXES): score += 2
    # Vaccine keywords in org name itself
    if any(k in name_lower for k in ['vaccine','vacc','vax','immuniz','health','medical']): score += 3
    # Anti-vax keywords in surrounding context
    if has_antivax_signal(ctx_lower): score += 3
    # Negative sentiment in context
    if sentiment_score(ctx_lower) < SENTIMENT_THRESHOLD: score += 2
    # "freedom" or "choice" (common in anti-vax naming)
    if any(w in name_lower for w in ['freedom','choice','rights','informed','natural']): score += 1

    return score


print('✓ NER + sentiment')
print(f'  Tracking {len(ANTIVAX_KEYWORDS)} anti-vax keywords')

## article fetcher

Grabs text from each URL. Strips nav, footer, and scripts first. Falls back to the full page if no article element is found.

In [ ]:
def fetch_article_text(url):
    """Fetch and extract main body text from a URL."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=12)
        if 'text/html' not in r.headers.get('Content-Type',''):
            return ''
        soup = BeautifulSoup(r.text, 'lxml')
        # Remove boilerplate
        for tag in soup(['script','style','nav','footer','header','aside','form']):
            tag.decompose()
        # Prefer article body
        body = soup.find('article') or soup.find('main') or soup.find('div', class_=re.compile('content|article|body|text', re.I))
        if body:
            return body.get_text(separator=' ', strip=True)[:8000]
        return soup.get_text(separator=' ', strip=True)[:8000]
    except Exception:
        return ''


print('✓ Article fetcher done')

## run

Phase 1 collects URLs from all three search sources. Phase 2 fetches each article and runs NER + sentiment on the text.

In [ ]:
# ── Collect all search results ──────────────────────────────────────────
all_results = []   # [{title, url, snippet, source}]
seen_urls   = set()

print('Phase 1: Collecting search results from DuckDuckGo + Bing + Wikipedia\n')

for query in tqdm(SEARCH_QUERIES, desc='Searching'):
    # DuckDuckGo
    for r in search_duckduckgo(query, max_results=RESULTS_PER_QUERY * MAX_PAGES):
        if r['url'] not in seen_urls:
            seen_urls.add(r['url'])
            all_results.append(r)

    time.sleep(DELAY_SECONDS)

    # Bing
    for r in search_bing(query, max_results=RESULTS_PER_QUERY):
        if r['url'] not in seen_urls:
            seen_urls.add(r['url'])
            all_results.append(r)

    time.sleep(DELAY_SECONDS)

    # Wikipedia
    for r in search_wikipedia(query):
        if r['url'] not in seen_urls:
            seen_urls.add(r['url'])
            all_results.append(r)

    time.sleep(DELAY_SECONDS * 0.5)

print(f'\n✓ Phase 1 complete: {len(all_results)} URLs collected')

In [ ]:
# ── Phase 2: NLP extraction over snippets + optionally full article text ─
org_mentions  = defaultdict(int)          # org_name → mention count
org_contexts  = defaultdict(list)         # org_name → [context snippets]
org_scores    = defaultdict(float)        # org_name → cumulative NLP score
org_urls      = defaultdict(set)          # org_name → source URLs

print(f'Phase 2: Running NER + sentiment on {len(all_results)} results...')
if SCRAPE_ARTICLES:
    print(f'   (getting full article text lol)')

for item in tqdm(all_results, desc='NLP analysis'):
    # Base text = title + snippet
    text = f"{item['title']} {item['snippet']}"

    # Optionally fetch full article
    if SCRAPE_ARTICLES and item['url'].startswith('http'):
        full_text = fetch_article_text(item['url'])
        if full_text:
            text += ' ' + full_text
        time.sleep(0.3)

    # Only process if anti-vax signal
    if not has_antivax_signal(text):
        continue

    # Extract ORG entities
    orgs_found = extract_orgs_ner(text)

    for org in orgs_found:
        score = score_org_candidate(org, text)
        if score >= 3:  # minimum NLP confidence
            org_mentions[org] += 1
            org_scores[org]   += score
            org_contexts[org].append(item['snippet'][:200])
            org_urls[org].add(item['url'])

print(f'\n✓ Phase 2 complete: {len(org_mentions)} organizations found')

## build results table

In [ ]:
# ── Results ───────────────────────────────────────────────────────
records = []
for org, count in org_mentions.items():
    if count >= MIN_CONFIDENCE:
        avg_score = org_scores[org] / count
        context   = org_contexts[org][0] if org_contexts[org] else ''
        source_urls = list(org_urls[org])[:3]
        records.append({
            'organization':    org,
            'mention_count':   count,
            'nlp_score':       round(avg_score, 2),
            'confidence':      'HIGH' if avg_score >= 7 else ('MEDIUM' if avg_score >= 5 else 'LOW'),
            'context_sample':  context[:250],
            'found_in_urls':   ' | '.join(source_urls),
            'discovered_at':   datetime.utcnow().isoformat()
        })

# Sorted by NLP score descending
df_discovered = pd.DataFrame(records).sort_values('nlp_score', ascending=False).reset_index(drop=True)

print(f'✓ {len(df_discovered)} organizations passed confidence threshold (min mentions: {MIN_CONFIDENCE})')
print(f'  HIGH confidence:   {len(df_discovered[df_discovered.confidence=="HIGH"])}')
print(f'  MEDIUM confidence: {len(df_discovered[df_discovered.confidence=="MEDIUM"])}')
print(f'  LOW confidence:    {len(df_discovered[df_discovered.confidence=="LOW"])}')

In [ ]:
# ── results table ────────────────────────────────────────
def highlight_confidence(val):
    colors = {
        'HIGH':   'background-color:#1a2e1a; color:#5cb85c; font-weight:bold',
        'MEDIUM': 'background-color:#2a2a1a; color:#e0c040',
        'LOW':    'background-color:#2a1a1a; color:#dc6b6b'
    }
    return colors.get(val, '')

display(df_discovered[['organization','mention_count','nlp_score','confidence','context_sample']]
    .style
    .applymap(highlight_confidence, subset=['confidence'])
    .bar(subset=['nlp_score'], color='#2a4a6a')
    .bar(subset=['mention_count'], color='#3a2a5a')
    .set_caption(f'Discovered Anti-Vaccine Organizations — {len(df_discovered)} total')
    .set_properties(**{'font-size':'12px'})
)

## stats

In [ ]:
print('=== Summary ===')
print(f'Total search results processed : {len(all_results)}')
print(f'URLs scraped            : {len(seen_urls)}')
print(f'Candidate orgs found (NER)     : {len(org_mentions)}')
print(f'Orgs passing confidence filter : {len(df_discovered)}')
print()

print('=== Top 100 ===')
print(df_discovered[['organization','nlp_score','mention_count','confidence']].head(100).to_string(index=False))

print()
print('=== Confidence ===')
print(df_discovered['confidence'].value_counts().to_string())

## export

Saves CSV and JSON to the same folder as this notebook. Download links appear inline after running.

In [ ]:
csv_path  = 'antivax_orgs_nlp.csv'
json_path = 'antivax_orgs_nlp.json'

# CSV
df_discovered.to_csv(csv_path, index=False)
print(f'✓ CSV saved: {csv_path}')
display(FileLink(csv_path, result_html_prefix='📥 Download CSV: '))

# JSON
output = {
    'generated': datetime.utcnow().isoformat(),
    'total_urls_scraped': len(all_results),
    'total_orgs_found': len(df_discovered),
    'config': {
        'queries': SEARCH_QUERIES,
        'results_per_query': RESULTS_PER_QUERY,
        'max_pages': MAX_PAGES,
        'min_confidence': MIN_CONFIDENCE,
        'sentiment_threshold': SENTIMENT_THRESHOLD
    },
    'organizations': df_discovered.to_dict(orient='records')
}
with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'✓ JSON saved: {json_path}')
display(FileLink(json_path, result_html_prefix='📥 Download JSON: '))

## filter

Change `SHOW_CONFIDENCE` to `HIGH`, `MEDIUM`, `LOW` otherwise leave it alone

In [ ]:
# ── Edit this ──────────────────────
SHOW_CONFIDENCE = 'HIGH'   # 'HIGH', 'MEDIUM', 'LOW', or '' for all
SHOW_TOP_N      = 50       # how many rows to show
# ───────────────────────────────────

filtered = df_discovered.copy()
if SHOW_CONFIDENCE:
    filtered = filtered[filtered['confidence'] == SHOW_CONFIDENCE]

print(f'Showing {min(len(filtered), SHOW_TOP_N)} organizations (confidence={SHOW_CONFIDENCE or "ALL"})')
display(filtered.head(SHOW_TOP_N)[['organization','mention_count','nlp_score','confidence','context_sample']].reset_index(drop=True))

In [ ]:
# Add extra queries here, then re-run cells from Step 7 forward
EXTRA_QUERIES = [
    # 'anti-vaccine groups Europe',
    # 'COVID vaccine opposition organizations',
    # 'anti-vax organizations Australia',
    # 'vaccine mandate opposition groups',
]

if EXTRA_QUERIES:
    SEARCH_QUERIES.extend(EXTRA_QUERIES)
    SEARCH_QUERIES = list(set(SEARCH_QUERIES))  # deduplicate
    print(f'✓ Added {len(EXTRA_QUERIES)} extra queries. Total: {len(SEARCH_QUERIES)}')
    print('Now re-run from Step 7 to include these in the search.')
else:
    print('No extra queries added. Uncomment lines above to add some.')